In [1]:
# activity data
import pandas as pd
import numpy as np
import os 
import warnings
warnings.filterwarnings('ignore')

from sklearn.preprocessing import StandardScaler

from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neural_network import MLPClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import f1_score,confusion_matrix,precision_score,recall_score
from sklearn.model_selection import cross_val_score
from sklearn.preprocessing import StandardScaler

from imblearn.over_sampling import RandomOverSampler

import matplotlib.pyplot as plt
import seaborn as sns
import shap

from utils.data_utils import correct_col_type,gen_date_col,transform_category_to_counts,min_max_perpatient

In [2]:
# Please change the path with the path of your dataset
DPATH = '../Dataset/'

# Read all tables into data_dict

files = os.listdir(DPATH)
data_dict = {}
summaries = {}
for f in files:
    if 'csv' not in f:
        continue
    print(f)
    fpth = os.path.join(DPATH,f)
    df = pd.read_csv(fpth)

    for col in df.columns:
        df[col] = correct_col_type(df,col)
    if 'date' in df.columns:
        df = df.rename(columns={'date':'timestamp'})
                
    fname = f.split('.')[0]
    data_dict[fname] = df

#  label data

In [3]:
## Generate a date column for Labels and Activity table 
lbl_df = gen_date_col(data_dict['Labels'],tcol='timestamp')
lbl_df

# activity data - 6 hour window

In [4]:
def assign_time_window(hour):
    if 0 <= hour < 6:
        return 'i'
    elif 6 <= hour < 12:
        return 'ii'
    elif 12 <= hour < 18:
        return 'iii'
    else:
        return 'iv'

In [5]:
act_df1 = gen_date_col(data_dict['Activity'],tcol='timestamp')
act_df1['time_window'] = act_df1['timestamp'].dt.hour.apply(assign_time_window)

act_df = transform_category_to_counts(act_df1,col='location_name',keys=['patient_id','date','time_window'])


act_df.to_csv('../output/data_cleaned/activity_count_6h_window.csv', index=False)
act_df